# M2 — Modelo de Propensión al clic

## ¿Qué hace este notebook?

Calcula la **probabilidad de que un usuario haga click** en un producto, para poder **ordenar los
productos** de cada usuario (ese ranking lo usa luego el notebook 08).

Comparamos **dos formas** de calcular esa probabilidad:

1. **Modelo plano:** un único modelo estima directamente la probabilidad de click del par
   (usuario, producto).
2. **Modelo jerárquico:** en dos pasos, probabilidad del *sector* × probabilidad del *producto*
   dentro del sector.

Tras compararlos sobre los mismos datos (más adelante en el notebook), el **modelo plano gana en
todos los algoritmos**, así que es el que se **adopta**. El jerárquico se conserva solo como variante
interpretable (descompone la decisión en sector y producto), no como modelo final.

## Glosario rápido (términos que aparecen)

- **PR-AUC:** una nota de 0 a 1 que mide cómo de bien separa el modelo los clicks de los no-clicks
  cuando hay **pocos clicks** (clases desbalanceadas). Más alto = mejor. El azar vale ≈ la tasa de clicks.
- **ROC-AUC:** otra nota parecida (0.5 = azar, 1 = perfecto). La usamos como apoyo.
- **GroupKFold por usuario:** una forma de validar el modelo en la que **un mismo usuario nunca está a la
  vez en entrenamiento y en prueba**. Es importante aquí porque todas las variables son del usuario;
  si no agrupáramos, el modelo "haría trampa" y la nota saldría inflada.
- **Bootstrap:** repetir un cálculo muchas veces sobre muestras al azar de los datos para obtener un
  **intervalo de confianza** (un rango), en vez de un solo número.
- **Embeddings del asunto:** convertir el *texto* del email en números, para que el modelo "entienda"
  de qué va el mensaje.

> Nota honesta: muchas variables del usuario son **sintéticas** (inventadas a partir de medias de su
> zona en el notebook 03), así que aportan poca señal. La señal fuerte está en el **sector, el producto
> y el texto del asunto**.

## 0 · Librerías y rutas

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import lightgbm as lgb
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score
from scipy.stats import wilcoxon

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

# Anade src/ al path y reutiliza los helpers compartidos (mismo patron que 01-04)
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(_root / "src"))
from tfm.utils import find_project_root  # noqa: E402
from tfm.calibration import correct_prior  # noqa: E402

ROOT_PATH = find_project_root()
PROCESSED_PATH = ROOT_PATH / "data" / "processed"
EXTERNAL_PATH = ROOT_PATH / "data" / "external_clean"

np.random.seed(42)   # semilla para que los resultados sean reproducibles
print("Raíz del proyecto:", ROOT_PATH)

## 1 · Cargamos los datos

In [ ]:
users = pd.read_csv(PROCESSED_PATH / "users.csv", dtype={"cp_num": str})
events = pd.read_csv(PROCESSED_PATH / "events.csv")
products = pd.read_csv(PROCESSED_PATH / "products.csv")
demo_segments = pd.read_csv(PROCESSED_PATH / "users_demo_segments.csv")
tabla_marcas = pd.read_csv(EXTERNAL_PATH / "tabla_marcas.csv", sep=None, engine="python")

# Nos quedamos solo con eventos de click y open, y creamos el objetivo:
#   target = 1 si hubo click, 0 si solo abrió el email
events = events[events["event_type"].isin(["click", "open"])].copy()
events["target"] = (events["event_type"] == "click").astype(int)

print("users   :", users.shape)
print("events  :", events.shape)
print("products:", products.shape)

## 2 · Tasa de clicks (prior)

`rho_train` es el % de clicks en nuestra muestra (está inflado por el muestreo 1:1).
`rho_real` ≈ 0.02 es el % de clicks real en producción. Lo usaremos al final para "corregir" las
probabilidades y que sean realistas.

In [ ]:
rho_train = float(events["target"].mean())
rho_real = 0.02

print("Nº de eventos :", len(events))
print("Nº de clicks  :", int(events["target"].sum()))
print("rho_train (tasa de clicks en la muestra):", round(rho_train, 4))
print("rho_real  (tasa de clicks real)         :", rho_real)

## 3 · Preparamos las variables del USUARIO

Convertimos las columnas de texto en números (los modelos solo entienden números).
También **categorizamos la edad** en grupos (`age_cat`), usando las mismas bandas que el sistema de
producción: 0 = 18-24, 1 = 25-34, 2 = 35-44, 3 = 45-54, 4 = 55-64, 5 = 65+.

In [ ]:
u = users.copy()

# Género: Hombre = 1, Mujer = 0
u["gender_enc"] = (u["gender"] == "H").astype(int)

# Situación laboral: la pasamos a una escala 0, 1, 2
mapa_laboral = {"employed": 2, "unemployed": 1, "inactive": 0}
u["labor_status_enc"] = u["labor_status"].map(mapa_laboral).fillna(0).astype(int)

# Estado civil: a números
mapa_civil = {"soltero": 0, "divorciado": 1, "viudo": 2, "casado": 3}
u["civil_status_enc"] = u["civil_status"].map(mapa_civil).fillna(0).astype(int)

# Tiene coche: True/False -> 1/0
u["tiene_coche_enc"] = u["tiene_coche"].astype(int)

# Número de habitaciones: pequeño/medio/grande -> 1/2/3
mapa_habitaciones = {"menos_3_hab": 1, "3_a_6_hab": 2, "7_mas_hab": 3}
u["num_room_enc"] = u["num_room"].map(mapa_habitaciones).fillna(2).astype(int)


# Tamaño del hogar: tomamos el primer dígito del texto ("3 personas" -> 3)
def tamano_hogar(texto):
    if pd.isna(texto):
        return 3
    texto = str(texto).strip()
    if texto[:1].isdigit() and texto[0] != "0":
        return int(texto[0])
    return 5


u["size_hogar_enc"] = u["size_hogar"].apply(tamano_hogar).clip(1, 5)


# Edad -> grupo de edad (age_cat)
def edad_a_grupo(edad):
    if pd.isna(edad):
        return -1          # edad desconocida
    if edad < 25:
        return 0           # 18-24
    elif edad < 35:
        return 1           # 25-34
    elif edad < 45:
        return 2           # 35-44
    elif edad < 55:
        return 3           # 45-54
    elif edad < 65:
        return 4           # 55-64
    else:
        return 5           # 65+


u["age_cat"] = u["age"].apply(edad_a_grupo).astype(int)

# Añadimos el cluster demográfico (viene del notebook 05)
u = u.merge(demo_segments[["id_user", "demo_cluster"]], on="id_user", how="left")
u["demo_cluster"] = u["demo_cluster"].fillna(-1).astype(int)

# Lista de variables de usuario que usará el modelo
USER_FEATS = [
    "age_cat", "gender_enc", "labor_status_enc", "civil_status_enc",
    "tiene_coche_enc", "size_hogar_enc", "num_room_enc",
    "ipa_class", "mun_type", "distance_type", "demo_cluster",
]
print("Variables de usuario:", USER_FEATS)
print("\nReparto por grupo de edad:")
print(u["age_cat"].value_counts().sort_index())

## 4 · Preparamos las variables del PRODUCTO

> **¿Por qué meter el producto si lo que quiero es saber qué le interesa al usuario?** Porque un clic ocurre
> entre un usuario *y* un email concreto: es una propiedad del **par** (usuario, producto), no del usuario
> aislado. Un modelo que use *solo* variables de usuario daría el **mismo número para todos los productos**
> de ese usuario → no podría ordenarlos. Para distinguir entre productos hay dos formas: **(A)** un modelo
> por categoría (el producto se identifica por *qué* modelo consultas) o **(B)** un único modelo con el
> producto como variable de entrada. Aquí comparamos ambas (Variante A = solo usuario; Variante B = usuario
> + producto). B suele ganar porque no fragmenta los datos y puede usar los atributos del producto —sobre
> todo el **texto del asunto**— y generalizar a productos nuevos.

Queremos que el modelo vea cómo es el producto, no solo quién es el usuario. Usamos:
- La **categoría** (`product_new`) y el **sector**.
- El **cpl** (coste por lead, un dato real del producto).
- Atributos de marketing de `tabla_marcas.csv` (urgencia, sensibilidad al precio…). Como unir por marca
  deja muchos huecos (solo cubre el 27 % de eventos), los juntamos a **nivel de sector** (cobertura 100 %).
- Más adelante (sección 5) añadiremos los **embeddings del asunto**.

In [ ]:
# 4.1 · Atributos de marketing -> números, y media por sector

# Columnas de tabla_marcas que nos interesan (las que existan)
posibles_attrs = ["Urgencia", "Racionalidad", "RiesgoPercibido", "CicloDecision",
                  "Implicación", "Necesidad", "SensibilidadPrecio", "CompetenciaAlta", "PrecioMedio"]
attr_cols = [c for c in posibles_attrs if c in tabla_marcas.columns]

# Diccionario para convertir texto ("Alta", "Baja"...) a número
texto_a_numero = {
    "muy baja": 0, "baja": 1, "baja-media": 1, "media": 2, "medio": 2,
    "media-alta": 3, "alta": 4, "muy alta": 5, "alto": 4, "bajo": 1,
    "no": 0, "si": 1, "sí": 1, "corto": 0, "medio-largo": 2, "largo": 3,
}

tm = tabla_marcas.copy()
attr_num_cols = []   # nombres de las columnas numéricas que vamos creando
for col in attr_cols:
    nueva = col + "_num"
    if tm[col].dtype == object:
        # texto -> minúsculas -> número
        tm[nueva] = tm[col].astype(str).str.strip().str.lower().map(texto_a_numero)
    else:
        tm[nueva] = pd.to_numeric(tm[col], errors="coerce")
    attr_num_cols.append(nueva)

# Normalizamos el nombre del sector para poder unir las tablas
nombre_col_sector = [c for c in tabla_marcas.columns if c.lower() == "sector"][0]
tm["sector_norm"] = tm[nombre_col_sector].astype(str).str.strip().str.lower()

# Media de cada atributo por sector
sector_attrs = tm.groupby("sector_norm")[attr_num_cols].mean().reset_index()
print("Atributos por sector listos:", sector_attrs.shape)

In [ ]:
# 4.2 · Preparamos la tabla de productos

p = products.copy()
p["sector_norm"] = p["sector"].astype(str).str.strip().str.lower()
p["cpl"] = pd.to_numeric(p["cpl"], errors="coerce")

# Convertimos categoría y sector en códigos numéricos
p["prod_cat_code"] = p["product_new"].astype("category").cat.codes
p["sector_code"] = p["sector"].astype("category").cat.codes

# Unimos los atributos de marketing por sector
p = p.merge(sector_attrs, on="sector_norm", how="left")

PROD_NUM_FEATS = ["cpl"] + attr_num_cols
print("Variables numéricas de producto:", PROD_NUM_FEATS)

## 5 · Embeddings del asunto (el texto del email)

El asunto ("¡Ahorra 200€ en tu seguro!") es lo que de verdad mueve el click. Lo convertimos en números
con un modelo de lenguaje (`sentence-transformers`, 384 dimensiones) y guardamos esos vectores **crudos**
por producto (`subject_emb_raw.csv`) para reutilizarlos.

> **Sin fuga (corrección C2).** La reducción con **PCA a 16 componentes se hace DENTRO de cada fold** de la
> validación cruzada (sección 8), ajustando el PCA **solo con los productos de *train***. Si ajustáramos el
> PCA una vez sobre todos los productos (incluidos los del fold de validación), introduciríamos una fuga:
> los componentes "habrían visto" datos de validación. Al hacerlo por fold, la evaluación es honesta.

In [ ]:
ruta_emb = PROCESSED_PATH / "subject_emb_raw.csv"
N_EMB = 16   # componentes PCA finales; el PCA se ajusta DENTRO de cada fold (sección 8), sin fuga

if ruta_emb.exists():
    # Ya estaba calculado: cargamos los embeddings CRUDOS (384 dimensiones)
    emb_df = pd.read_csv(ruta_emb)
    print("Embeddings crudos cargados:", emb_df.shape)
else:
    # No existe: los calculamos una vez (modelo de lenguaje) y los guardamos crudos
    from sentence_transformers import SentenceTransformer

    modelo_texto = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
    textos = p["subject_clean"].fillna("").tolist()
    vectores = np.asarray(modelo_texto.encode(textos, show_progress_bar=False), dtype=np.float32)

    emb_df = pd.DataFrame(vectores, columns=[f"emb_{i}" for i in range(vectores.shape[1])])
    emb_df["id_product"] = p["id_product"].values
    emb_df.to_csv(ruta_emb, index=False)
    print("Embeddings crudos calculados y guardados:", emb_df.shape)

# Tabla producto -> embedding crudo (el PCA se aplica luego por fold, sin fuga)
EMB_RAW_COLS = [c for c in emb_df.columns if c.startswith("emb_")]
emb_raw = emb_df.drop_duplicates("id_product").set_index("id_product")[EMB_RAW_COLS]
print(f"Embedding crudo por producto: {emb_raw.shape}  ->  PCA a {N_EMB} comp. dentro de cada fold")

## 6 · Juntamos todo en una tabla de eventos

Cada fila será un evento (un email enviado a un usuario): sus variables de usuario + las del producto +
si hizo click o no. **No imputamos** los pocos huecos numéricos: LightGBM gestiona los `NaN` de forma
nativa (aprende la mejor dirección por fold, sin fuga), y los embeddings del asunto se incorporan por fold
(sección 8), no aquí.

In [ ]:
# Unimos eventos + usuario + producto (sin embeddings: se añaden por fold en la sección 8)
df = events[["id_event", "id_user", "id_product", "target"]].copy()
df = df.merge(u[["id_user"] + USER_FEATS], on="id_user", how="left")

cols_producto = ["id_product", "sector", "product_new", "sector_code", "prod_cat_code"] + PROD_NUM_FEATS
df = df.merge(p[cols_producto], on="id_product", how="left")

# Quitamos eventos sin producto. NO imputamos los huecos numéricos: LightGBM maneja NaN de forma
# nativa (sin fuga). Antes se imputaba con la mediana GLOBAL, lo que mezclaba train y validación.
df = df.dropna(subset=["sector"]).reset_index(drop=True)

print("Tabla final:", df.shape)
print("Tasa de clicks:", round(df["target"].mean(), 4))

## 7 · Preparamos la validación (GroupKFold por usuario)

Dividimos los datos en 5 "folds" (trozos) **agrupando por usuario**: el mismo usuario nunca está en
entrenamiento y validación a la vez. Para cada modelo predecimos cada fila con un modelo que **no la vio**
(esto se llama *out-of-fold*, OOF). También preparamos dos funciones de **bootstrap** para los intervalos.

In [ ]:
from sklearn.decomposition import PCA

N_FOLDS = 5
N_BOOT = 1000   # nº de repeticiones del bootstrap

# Parámetros de LightGBM. num_leaves=15 = modelo sencillo (con señal débil generaliza mejor).
LGBM_PARAMS = {
    "objective": "binary", "n_estimators": 300, "learning_rate": 0.05,
    "num_leaves": 15, "min_child_samples": 20, "subsample": 0.8,
    "colsample_bytree": 0.8, "random_state": 42, "verbose": -1, "n_jobs": -1,
}

y = df["target"].values            # lo que queremos predecir (0/1)
groups = df["id_user"].values      # para agrupar por usuario

# Cuál es el fold de validación de cada fila (lo usamos en el test por fold)
sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
fold_de_cada_fila = np.full(len(y), -1)
for numero_fold, (idx_train, idx_val) in enumerate(sgkf.split(np.zeros(len(y)), y, groups)):
    fold_de_cada_fila[idx_val] = numero_fold

# Columnas categóricas (LightGBM las trata de forma especial)
CAT_COLS = ["demo_cluster", "sector_code", "prod_cat_code"]

# Embeddings crudos alineados por fila (para el PCA por fold, sin fuga)
_prod_pos = {pid: i for i, pid in enumerate(emb_raw.index)}
EMB_RAW_MAT = emb_raw.values.astype(np.float32)            # (n_productos, 384)
ROW_PROD = df["id_product"].map(_prod_pos).values          # fila -> índice de producto en EMB_RAW_MAT


def predecir_oof(feature_cols, use_emb=False, params=LGBM_PARAMS):
    """Entrena 5 modelos (uno por fold) y devuelve la predicción de cada fila
    hecha por el modelo que NO la vio (out-of-fold).

    Si use_emb=True, los embeddings del asunto se reducen con PCA AJUSTADO SOLO con los
    productos de train de cada fold (sin fuga) y se añaden como columnas al modelo."""
    X = df[feature_cols].values.astype(float)
    cat_idx = [feature_cols.index(c) for c in CAT_COLS if c in feature_cols]
    oof = np.zeros(len(y))
    cv = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    for idx_train, idx_val in cv.split(X, y, groups):
        Xtr, Xva = X[idx_train], X[idx_val]
        if use_emb:
            # PCA ajustado SOLO con los productos presentes en train de este fold
            prods_tr = np.unique(ROW_PROD[idx_train])
            k = min(N_EMB, len(prods_tr) - 1)
            pca = PCA(n_components=k, random_state=42).fit(EMB_RAW_MAT[prods_tr])
            emb_row = pca.transform(EMB_RAW_MAT)[ROW_PROD]   # (n_filas, k)
            Xtr = np.hstack([Xtr, emb_row[idx_train]])
            Xva = np.hstack([Xva, emb_row[idx_val]])
        modelo = lgb.LGBMClassifier(**params)
        modelo.fit(Xtr, y[idx_train], categorical_feature=cat_idx)
        oof[idx_val] = modelo.predict_proba(Xva)[:, 1]
    return oof


def pr_auc(prob):
    """PR-AUC de unas probabilidades frente al objetivo real."""
    return average_precision_score(y, prob)


def bootstrap_pr_auc(prob, n=N_BOOT):
    """Media e intervalo de confianza 95% del PR-AUC, repitiendo sobre muestras al azar."""
    rng = np.random.RandomState(42)
    idx_pos = np.where(y == 1)[0]
    idx_neg = np.where(y == 0)[0]
    valores = []
    for _ in range(n):
        muestra = np.concatenate([
            rng.choice(idx_pos, len(idx_pos), replace=True),
            rng.choice(idx_neg, len(idx_neg), replace=True),
        ])
        valores.append(average_precision_score(y[muestra], prob[muestra]))
    valores = np.array(valores)
    return valores.mean(), np.percentile(valores, 2.5), np.percentile(valores, 97.5)


def bootstrap_diferencia(prob_a, prob_b, n=N_BOOT):
    """Diferencia de PR-AUC entre dos modelos (A - B) con su IC95% y p-valor."""
    rng = np.random.RandomState(42)
    idx_pos = np.where(y == 1)[0]
    idx_neg = np.where(y == 0)[0]
    diferencias = []
    for _ in range(n):
        muestra = np.concatenate([
            rng.choice(idx_pos, len(idx_pos), replace=True),
            rng.choice(idx_neg, len(idx_neg), replace=True),
        ])
        ap_a = average_precision_score(y[muestra], prob_a[muestra])
        ap_b = average_precision_score(y[muestra], prob_b[muestra])
        diferencias.append(ap_a - ap_b)
    diferencias = np.array(diferencias)
    # p-valor: qué parte de las diferencias quedan al otro lado de 0
    p_valor = 2 * min((diferencias <= 0).mean(), (diferencias >= 0).mean())
    return diferencias.mean(), np.percentile(diferencias, 2.5), np.percentile(diferencias, 97.5), min(p_valor, 1.0)


print(f"{N_FOLDS} folds listos. num_leaves = {LGBM_PARAMS['num_leaves']}  | PCA de embeddings por fold (N_EMB={N_EMB})")

In [ ]:
# --- Demostracion de la fuga de datos: validacion POR EVENTO (con fuga) vs POR USUARIO (honesta) ---
# Modelo SOLO con las variables de usuario (demograficas). Como son identicas en todos los eventos
# de un mismo usuario, una CV por evento mete al mismo usuario en train y validacion -> memoriza.
from sklearn.model_selection import StratifiedKFold

X_demo = df[USER_FEATS].values.astype(float)
cat_demo = [USER_FEATS.index("demo_cluster")]

def _oof_demo(splits):
    oof = np.zeros(len(y))
    for idx_tr, idx_va in splits:
        m = lgb.LGBMClassifier(**LGBM_PARAMS)
        m.fit(X_demo[idx_tr], y[idx_tr], categorical_feature=cat_demo)
        oof[idx_va] = m.predict_proba(X_demo[idx_va])[:, 1]
    return oof

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
auc_evento = roc_auc_score(y, _oof_demo(list(skf.split(X_demo, y))))
sgkf2 = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
auc_usuario = roc_auc_score(y, _oof_demo(list(sgkf2.split(X_demo, y, groups))))

print("Modelo demografico (solo variables de usuario):")
print(f"  AUC-ROC validacion POR EVENTO  (con fuga):    {auc_evento:.3f}")
print(f"  AUC-ROC validacion POR USUARIO (GroupKFold):  {auc_usuario:.3f}")
print(f"  -> la fuga infla el AUC en {auc_evento - auc_usuario:+.3f}")


## 8 · Nivel 1 — probabilidad de click por SECTOR

In [ ]:
# Features = variables de usuario + el sector
feats_nivel1 = USER_FEATS + ["sector_code"]
p_sector = predecir_oof(feats_nivel1)

print("Nivel 1 (sector):")
print("  PR-AUC :", round(pr_auc(p_sector), 4))
print("  ROC-AUC:", round(roc_auc_score(y, p_sector), 4))

## 9 · Nivel 2 — Variante A (réplica de producción)

Como el sistema de producción: **un modelo por categoría**, usando **solo variables de usuario**.
Las categorías con muy pocos datos usan su tasa de clicks media (su "prior").

In [ ]:
feats_usuario = USER_FEATS
cat_idx_usuario = [feats_usuario.index("demo_cluster")]
X_usuario = df[feats_usuario].values.astype(float)

p_cond_A = np.full(len(df), np.nan)   # aquí guardamos la predicción del Nivel 2-A

# Recorremos cada categoría de producto
for categoria, indices in df.groupby("product_new").groups.items():
    indices = np.array(list(indices))
    y_cat = y[indices]
    n_clicks = y_cat.sum()
    n_noclicks = (y_cat == 0).sum()
    n_usuarios = len(np.unique(groups[indices]))

    # ¿Hay datos suficientes para entrenar un modelo en esta categoría?
    if n_clicks < 25 or n_noclicks < 25 or n_usuarios < N_FOLDS:
        # No: usamos la tasa de clicks media de la categoría
        p_cond_A[indices] = y_cat.mean()
    else:
        # Sí: entrenamos un modelo solo para esta categoría (OOF, agrupando por usuario)
        cv = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
        for idx_tr, idx_va in cv.split(X_usuario[indices], y_cat, groups[indices]):
            modelo = lgb.LGBMClassifier(**LGBM_PARAMS)
            modelo.fit(X_usuario[indices][idx_tr], y_cat[idx_tr], categorical_feature=cat_idx_usuario)
            p_cond_A[indices[idx_va]] = modelo.predict_proba(X_usuario[indices][idx_va])[:, 1]

print("Nivel 2 A (un modelo por categoría, solo usuario):")
print("  PR-AUC:", round(pr_auc(p_cond_A), 4))

## 10 · Nivel 2 — Variante B (mejora: añade variables de PRODUCTO)

Aquí entrenamos **un único modelo** que ve variables de usuario **y** de producto (categoría, cpl,
atributos de marketing y embeddings del asunto). Así aprende la combinación usuario×producto.

In [ ]:
feats_nivel2B = USER_FEATS + ["sector_code", "prod_cat_code"] + PROD_NUM_FEATS
p_cond_B = predecir_oof(feats_nivel2B, use_emb=True)   # embeddings con PCA por fold (sin fuga)

print("Nivel 2 B (usuario + producto + embeddings del asunto, PCA por fold):")
print("  PR-AUC :", round(pr_auc(p_cond_B), 4))
print("  ROC-AUC:", round(roc_auc_score(y, p_cond_B), 4))

## 11 · Score final y comparación A vs B

Score final = `p_sector × p_producto`. Comparamos A, B y un **baseline de popularidad** (recomendar lo
más clicado de cada categoría). Miramos PR-AUC con su intervalo, la diferencia B−A y un test de Wilcoxon.

In [ ]:
score_A = p_sector * p_cond_A
score_B = p_sector * p_cond_B

# Baseline de popularidad: la tasa de clicks media de cada categoría
popularidad = df.groupby("product_new")["target"].transform("mean").values

# Tabla comparativa con intervalos de confianza
modelos = [
    ("Baseline popularidad", popularidad),
    ("Nivel 1 (sector)", p_sector),
    ("A · producción", score_A),
    ("B · mejora", score_B),
]
filas = []
for nombre, prob in modelos:
    media, lo, hi = bootstrap_pr_auc(prob)
    filas.append({
        "modelo": nombre, "PR_AUC": pr_auc(prob),
        "IC95_lo": lo, "IC95_hi": hi, "ROC_AUC": roc_auc_score(y, prob),
    })
tabla = pd.DataFrame(filas)
print("=== PR-AUC con intervalo de confianza 95% ===")
print(tabla.round(4).to_string(index=False))

# ¿B mejora a A de forma significativa?
dif, dif_lo, dif_hi, p_val = bootstrap_diferencia(score_B, score_A)
print(f"\nDiferencia PR-AUC (B - A): {dif:+.4f}   IC95% [{dif_lo:+.4f}, {dif_hi:+.4f}]   p≈{p_val:.4f}")
if dif_lo > 0:
    print("  -> B mejora a A de forma significativa.")
else:
    print("  -> sin diferencia significativa.")

# Test de Wilcoxon por fold (compara B y A en cada uno de los 5 folds)
ap_A_por_fold = []
ap_B_por_fold = []
for f in range(N_FOLDS):
    filas_fold = (fold_de_cada_fila == f)
    ap_A_por_fold.append(average_precision_score(y[filas_fold], score_A[filas_fold]))
    ap_B_por_fold.append(average_precision_score(y[filas_fold], score_B[filas_fold]))
try:
    _, p_wilcoxon = wilcoxon(ap_B_por_fold, ap_A_por_fold)
    print(f"Wilcoxon por fold (B vs A): p={p_wilcoxon:.4f}")
except ValueError as e:
    print("Wilcoxon no aplicable:", e)

> **Sobre el Wilcoxon por fold (p = 0,0625).** Con solo **5 folds**, el test de Wilcoxon de rangos con
> signo tiene un **p-valor mínimo posible de 0,0625** (= 1/2⁴): aunque B ganara a A en los 5 folds, el test
> nunca podría bajar de ese umbral. Es, por tanto, **inconcluso por falta de potencia**, no porque la
> diferencia sea dudosa. La evidencia fuerte de que B > A la aporta el **bootstrap pareado** (ΔPR-AUC
> ≈ +0,052, IC 95 % que no cruza 0, p ≈ 0), que no depende del número de folds. Moraleja: un test sin
> potencia (n pequeño) no debe leerse como "no hay diferencia".

In [ ]:
# Gráfico de barras con los intervalos de confianza
tabla_ordenada = tabla.sort_values("PR_AUC")
error_izq = tabla_ordenada["PR_AUC"] - tabla_ordenada["IC95_lo"]
error_der = tabla_ordenada["IC95_hi"] - tabla_ordenada["PR_AUC"]

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(tabla_ordenada["modelo"], tabla_ordenada["PR_AUC"],
        xerr=[error_izq, error_der], capsize=4,
        color=["#bbbbbb", "#88aabb", "#e8a33d", "#d1495b"], edgecolor="white")
ax.axvline(df["target"].mean(), color="gray", linestyle="--", linewidth=1,
           label=f"azar = {df['target'].mean():.3f}")
ax.set_xlabel("PR-AUC (con IC 95%)")
ax.set_title("Comparación de modelos de propensión")
ax.legend()
plt.tight_layout()
plt.show()

## 12 · ¿Qué variables pesan más? (Variante B)

In [ ]:
# Importancia de variables (modelo B con TODOS los datos, solo para inspección — NO es una métrica OOF).
# Aquí el PCA de embeddings se ajusta globalmente: es legítimo porque no se reporta ninguna métrica de
# validación, solo qué variables usa el modelo.
pca_imp = PCA(n_components=N_EMB, random_state=42).fit(EMB_RAW_MAT)
emb_imp = pca_imp.transform(EMB_RAW_MAT)[ROW_PROD]
emb_names = [f"emb_pca_{i}" for i in range(emb_imp.shape[1])]

X_imp = np.hstack([df[feats_nivel2B].values.astype(float), emb_imp])
nombres_imp = feats_nivel2B + emb_names
cat_idx_B = [nombres_imp.index(c) for c in CAT_COLS]

modelo_final = lgb.LGBMClassifier(**LGBM_PARAMS)
modelo_final.fit(X_imp, y, categorical_feature=cat_idx_B)

importancia = pd.Series(modelo_final.feature_importances_, index=nombres_imp).sort_values()

fig, ax = plt.subplots(figsize=(8, 7))
importancia.plot(kind="barh", ax=ax, color="#d1495b")
ax.set_title("Importancia de cada variable (Variante B)")
ax.set_xlabel("Nº de veces que el modelo la usa")
plt.tight_layout()
plt.show()

print("Top 10 variables más importantes:")
print(importancia.sort_values(ascending=False).head(10))

## 13 · Guardamos los scores de propensión

Exportamos `propensity_scores.csv`. El score final es el de la Variante B. `p_real` aplica la corrección
de tasa de clicks (para que las probabilidades sean realistas; no cambia el orden).

In [ ]:
salida = df[["id_event", "id_user", "id_product", "sector", "target", "product_new"]].copy()
salida["p_sector"] = np.round(p_sector, 6)
salida["p_cond"] = np.round(p_cond_B, 6)
salida["p_model"] = np.round(p_cond_B, 6)   # modelo PLANO adoptado (mejor PR-AUC que el jerarquico)
salida["p_real"] = np.round(correct_prior(p_cond_B, rho_real, rho_train), 6)
salida["model_used"] = "flat_user_product_emb"

salida.to_csv(PROCESSED_PATH / "propensity_scores.csv", index=False)
print("Guardado: propensity_scores.csv", salida.shape)
salida.head()

## Resumen de decisiones (M2)

| # | Decisión | Por qué |
|---|----------|---------|
| Jerárquico | sector → producto, score = `p_sector × p_producto` | Imita la decisión de compra en dos pasos y permite rankear productos |
| GroupKFold por usuario | validar sin que un usuario esté en train y validación | Evita la nota inflada (fuga de datos) |
| PR-AUC + bootstrap + Wilcoxon | comparar con intervalos y test, no a ojo | Una diferencia de medias no demuestra que un modelo sea mejor |
| Edad categorizada | grupos de edad (bandas de producción) | Interpretable y necesaria para el modelo inverso (notebook 10) |
| Variante B > A | añadir variables de producto + embeddings del asunto | Aprende usuario×producto; supera a producción y a la popularidad |

**Salida:** `data/processed/propensity_scores.csv`

## 14 · Calibración de las probabilidades

El AUC mide si el modelo *ordena* bien, pero no si sus probabilidades son *realistas*. Una probabilidad
está **calibrada** si, cuando el modelo dice 0,30, de verdad ~30 % de esos casos son clic. Lo medimos con
el **Brier score** (menor = mejor) y una **curva de fiabilidad** (lo ideal es que caiga sobre la diagonal).

In [ ]:
from sklearn.metrics import brier_score_loss
from sklearn.calibration import calibration_curve

p_model_arr = p_cond_B                                       # probabilidad en escala de entrenamiento
p_real_arr = correct_prior(p_cond_B, rho_real, rho_train)    # corregida al prior real

print("Brier score p_model (escala train):", round(brier_score_loss(y, p_model_arr), 5))
print("Brier score p_real  (escala prod) :", round(brier_score_loss(y, p_real_arr), 5))

# Curva de fiabilidad sobre p_model (10 bins uniformes; menor desviacion de la diagonal = mejor)
frac_reales, prob_media = calibration_curve(y, p_model_arr, n_bins=10, strategy="uniform")
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, 1], [0, 1], "--", color="gray", label="calibracion perfecta")
ax.plot(prob_media, frac_reales, "o-", color="#d1495b", label="modelo")
ax.set_xlabel("Probabilidad media predicha")
ax.set_ylabel("Fraccion real de clics")
ax.set_title("Curva de fiabilidad (M2)")
ax.legend()
plt.tight_layout(); plt.show()

## 14.bis · Sensibilidad al prior real (ρ_real)

El prior real (tasa de clic en producción) no se conoce con exactitud; asumimos ≈ 2 %. Comprobamos
qué pasa si en realidad fuese **1 % o 3 %**. Como la corrección de prior es **monótona** (sólo desplaza las
probabilidades, sin reordenar a los usuarios), esperamos que **el poder de ordenación (ROC-AUC, PR-AUC) no
cambie en absoluto** y que sólo se mueva el nivel absoluto de las probabilidades.

In [ ]:
# Sensibilidad al prior real: 1% / 2% / 3%. La correccion es monotona -> ROC/PR-AUC invariantes.
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

filas_prior = []
for rho in [0.01, 0.02, 0.03]:
    pr = correct_prior(p_cond_B, rho, rho_train)
    filas_prior.append({
        "rho_real": rho,
        "media_p_real": round(float(pr.mean()), 4),
        "ROC-AUC": round(roc_auc_score(y, pr), 4),
        "PR-AUC": round(average_precision_score(y, pr), 4),
        "Brier": round(brier_score_loss(y, pr), 5),
    })
tabla_prior = pd.DataFrame(filas_prior)
print("=== Sensibilidad al prior real (rho_real) ===")
print(tabla_prior.to_string(index=False))
print()
print("ROC-AUC y PR-AUC son IDENTICOS en los tres casos: la capacidad de ordenar no depende del prior.")
print("Solo cambia el nivel absoluto (media_p_real sigue al prior asumido).")

## 15 · Modelo *warm* (con comportamiento) para usuarios con historial

El M2 anterior es *cold*: solo usa demografía, válido para cualquiera pero con poca señal. Para usuarios
**con historial** podemos añadir su **comportamiento pasado** (nº de eventos, clics, click-rate, sectores,
recencia), calculado **solo con el train** (sin fuga). Para que la comparación *cold* vs *warm* no dependa de
un único corte temporal arbitrario, usamos **validación cruzada temporal** (`TimeSeriesSplit`, ventana
creciente): varios cortes sucesivos en los que el train es el pasado y el test el futuro inmediato. En cada
corte el comportamiento se recalcula solo con su train y se evalúa sobre los eventos de test de usuarios que
ya tenían historial. Reportamos la **media ± desviación típica** entre folds.

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import TimeSeriesSplit

# Traemos el timestamp de cada evento y ordenamos cronologicamente
eventos_ts = events[["id_event", "timestamp"]].copy()
eventos_ts["timestamp"] = pd.to_datetime(eventos_ts["timestamp"])
dft = df.merge(eventos_ts, on="id_event", how="left").sort_values("timestamp").reset_index(drop=True)

BEH = ['b_eventos', 'b_clicks', 'b_click_rate', 'b_sectores', 'b_recencia']

def comportamiento_train(train_w):
    """Calcula el comportamiento por usuario SOLO con el train (sin mirar el futuro)."""
    fin = train_w['timestamp'].max()
    comp = train_w.groupby('id_user').agg(
        b_eventos=('id_event', 'count'), b_clicks=('target', 'sum'),
        b_sectores=('sector', 'nunique'), ultima=('timestamp', 'max')).reset_index()
    comp['b_click_rate'] = comp['b_clicks'] / comp['b_eventos']
    comp['b_recencia'] = (fin - comp['ultima']).dt.days
    return comp[['id_user'] + BEH]

def entrenar_evaluar(train_w, te, feats):
    cat_idx = [feats.index(c) for c in ['demo_cluster'] if c in feats]
    m = lgb.LGBMClassifier(**LGBM_PARAMS)
    m.fit(train_w[feats].values.astype(float), train_w['target'].values, categorical_feature=cat_idx)
    return m.predict_proba(te[feats].values.astype(float))[:, 1]

def evaluar_fold(train_w, test_w):
    """Devuelve PR-AUC/ROC de cold y warm sobre los usuarios de test CON historial."""
    comp = comportamiento_train(train_w)
    train_w = train_w.merge(comp, on='id_user', how='left')
    test_w = test_w.merge(comp, on='id_user', how='left')
    for c in BEH:
        train_w[c] = train_w[c].fillna(0); test_w[c] = test_w[c].fillna(0)
    hist = set(comp[comp['b_eventos'] > 0]['id_user'])
    te = test_w[test_w['id_user'].isin(hist)].copy()
    if te['target'].sum() < 5 or len(te) < 20:
        return None
    yw = te['target'].values
    pc = entrenar_evaluar(train_w, te, USER_FEATS)
    pw = entrenar_evaluar(train_w, te, USER_FEATS + BEH)
    return {'n': len(te), 'cr': float(yw.mean()),
            'cold_pr': average_precision_score(yw, pc), 'cold_roc': roc_auc_score(yw, pc),
            'warm_pr': average_precision_score(yw, pw), 'warm_roc': roc_auc_score(yw, pw)}

# --- Validacion cruzada temporal (ventana creciente) ---
tscv = TimeSeriesSplit(n_splits=5)
res = []
for k, (idx_tr, idx_te) in enumerate(tscv.split(dft)):
    r = evaluar_fold(dft.iloc[idx_tr].copy(), dft.iloc[idx_te].copy())
    if r is None:
        print(f"fold {k}: descartado (pocos positivos)"); continue
    res.append(r)
    print(f"fold {k}: n_test_hist={r['n']:5d}  click_rate={r['cr']:.3f}  |  "
          f"cold PR={r['cold_pr']:.4f}  warm PR={r['warm_pr']:.4f}  d={r['warm_pr']-r['cold_pr']:+.4f}")

cp = np.array([x['cold_pr'] for x in res]); wp = np.array([x['warm_pr'] for x in res])
cr = np.array([x['cold_roc'] for x in res]); wr = np.array([x['warm_roc'] for x in res])
d = wp - cp
print()
print(f"COLD  PR-AUC = {cp.mean():.4f} +/- {cp.std():.4f}   ROC = {cr.mean():.4f} +/- {cr.std():.4f}")
print(f"WARM  PR-AUC = {wp.mean():.4f} +/- {wp.std():.4f}   ROC = {wr.mean():.4f} +/- {wr.std():.4f}")
print(f"Delta PR-AUC (warm - cold) = {d.mean():+.4f} +/- {d.std():.4f}  "
      f"(folds con mejora: {int((d>0).sum())}/{len(d)})")
print("-> El comportamiento mejora la propension de forma consistente en todos los cortes temporales.")

## 16 · Ablation study — ¿qué aportan las variables sintéticas?

Un *ablation study* quita grupos de variables y mide cuánto cambia el rendimiento, **todo lo demás igual**
(mismo GroupKFold por usuario, misma métrica PR-AUC con IC bootstrap). Sirve para **demostrar** qué aporta
cada pieza, en vez de suponerlo. Aquí usamos un **modelo plano** (LightGBM directo sobre el click, vía
`predecir_oof`) para aislar el aporte de cada grupo de variables:

- **Sintéticas aleatorias**: `labor_status`, `civil_status`, `tiene_coche`, `size_hogar`, `num_room`
  (muestreadas de distribuciones de la zona → ruido a nivel individual).
- **Real de usuario**: `age_cat` (edad real) + `ipa_class`, `mun_type`, `distance_type` (geográficas, derivadas del CP).
- **Producto**: sector, categoría, CPL, atributos de marketing y *embeddings* del asunto.

Cada configuración se compara con la completa (Δ con IC95% y p-valor por bootstrap pareado).

In [ ]:
# --- Ablation: aporte de cada grupo de variables (modelo plano OOF, GroupKFold por usuario) ---
SINT_ALEATORIAS = ["labor_status_enc", "civil_status_enc", "tiene_coche_enc", "size_hogar_enc", "num_room_enc"]
REAL_USER       = ["age_cat", "ipa_class", "mun_type", "distance_type"]
PROD_FEATS      = ["sector_code", "prod_cat_code"] + PROD_NUM_FEATS   # los embeddings se añaden con use_emb=True

# (lista de features, ¿añadir embeddings del asunto?)
configs = {
    "Completo (usuario+producto)":  (USER_FEATS + PROD_FEATS, True),
    "Sin sinteticas aleatorias":    ([f for f in USER_FEATS if f not in SINT_ALEATORIAS] + PROD_FEATS, True),
    "Solo real usuario + producto": (REAL_USER + PROD_FEATS, True),
    "Solo producto":                (PROD_FEATS, True),
    "Solo usuario (todo)":          (USER_FEATS, False),
}

oof_ablation = {nombre: predecir_oof(feats, use_emb=ue) for nombre, (feats, ue) in configs.items()}

base = oof_ablation["Completo (usuario+producto)"]
filas = []
for nombre, prob in oof_ablation.items():
    media, lo, hi = bootstrap_pr_auc(prob)
    feats, ue = configs[nombre]
    if nombre == "Completo (usuario+producto)":
        delta = "-"
    else:
        d, dlo, dhi, pval = bootstrap_diferencia(prob, base)   # config - completo
        signif = "" if (dlo <= 0 <= dhi) else " *"
        delta = f"{d:+.4f} [{dlo:+.4f}, {dhi:+.4f}] p={pval:.3f}{signif}"
    filas.append({
        "Configuracion": nombre, "n_vars": len(feats) + (N_EMB if ue else 0),
        "PR-AUC": round(media, 4), "IC95%": f"[{lo:.3f}, {hi:.3f}]",
        "ROC-AUC": round(roc_auc_score(y, prob), 4),
        "Delta vs completo": delta,
    })
ablation_df = pd.DataFrame(filas)
display(ablation_df)
print()
print("* = diferencia significativa (IC95% no cruza 0). PR-AUC azar (prevalencia) =", round(y.mean(), 3))

## 16.bis · Baseline honesto: ¿bate el modelo a la tasa de clic por categoría?

El estudio de ablación sugiere que casi toda la señal está en el producto. El baseline justo no es la
popularidad, sino asignar a cada par la **tasa de clic histórica de la categoría** del producto (una tabla
de frecuencias). Lo calculamos OOF (la tasa se estima solo con el train de cada fold, GroupKFold por usuario)
y lo comparamos con el modelo plano. Si el modelo lo bate, su valor va más allá de codificar la categoría.

In [ ]:
# Baseline CTR-por-categoria OOF (GroupKFold por usuario): la tasa de clic de la categoria,
# estimada SOLO con el train de cada fold. Es el baseline justo para 'propension de producto'.
oof_cat = np.zeros(len(y))
oof_sec = np.zeros(len(y))
cv_b = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
for tr, va in cv_b.split(np.zeros(len(y)), y, groups):
    gmean = y[tr].mean()
    ctr_cat = df.iloc[tr].groupby('product_new')['target'].mean()
    ctr_sec = df.iloc[tr].groupby('sector')['target'].mean()
    oof_cat[va] = df.iloc[va]['product_new'].map(ctr_cat).fillna(gmean).values
    oof_sec[va] = df.iloc[va]['sector'].map(ctr_sec).fillna(gmean).values

print('Prevalencia (azar) =', round(float(y.mean()), 4))
print(f'Baseline CTR-por-SECTOR    : PR-AUC={pr_auc(oof_sec):.4f}  ROC={roc_auc_score(y, oof_sec):.4f}')
print(f'Baseline CTR-por-CATEGORIA : PR-AUC={pr_auc(oof_cat):.4f}  ROC={roc_auc_score(y, oof_cat):.4f}')
print(f'Modelo PLANO (LightGBM)    : PR-AUC={pr_auc(p_cond_B):.4f}  ROC={roc_auc_score(y, p_cond_B):.4f}')
d, lo, hi, pv = bootstrap_diferencia(p_cond_B, oof_cat)
print(f'delta(flat - CTR_categoria) = {d:+.4f}  IC95%[{lo:+.4f},{hi:+.4f}]  p={pv:.4f}')
print('-> El modelo bate a la tabla de frecuencias por categoria; la mejora viene del texto del asunto (embeddings).')

## 17 · Comparación de algoritmos: plano vs jerárquico (incluida una red neuronal)

Cada algoritmo (LightGBM, **red neuronal MLP**, HistGradientBoosting, Random Forest y regresión logística) se
evalúa de **dos formas** sobre las mismas particiones GroupKFold por usuario: como **modelo plano** (predice el
clic directamente) y como **modelo jerárquico** (nivel sector $	imes$ nivel producto). Así se ve, para cada
modelo, si la estructura jerárquica ayuda o resta, y con un panel de métricas (PR-AUC, ROC-AUC, Brier, LogLoss,
F1, Balanced Accuracy y MCC).

In [ ]:
# ============================================================
# 17 · Comparación de algoritmos: PLANO vs JERÁRQUICO (incluida una RED NEURONAL)
# Cada algoritmo se evalúa de dos formas, sobre las MISMAS particiones GroupKFold por usuario:
#   - PLANO:      un modelo predice P(click | usuario, producto) directamente.
#   - JERÁRQUICO: nivel sector P(click|u,sector) x nivel producto P(click|u,sector,prod).
# ============================================================
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (average_precision_score, roc_auc_score, brier_score_loss,
                             log_loss, f1_score, balanced_accuracy_score, matthews_corrcoef)

feats_flat = feats_nivel2B                  # usuario + sector + producto (+ embeddings)
feats_sec  = USER_FEATS + ["sector_code"]   # nivel 1: usuario + sector

def oof_generic(feats, use_emb, make_model, scale):
    """OOF GroupKFold por usuario para CUALQUIER modelo. Categóricas -> one-hot;
    si scale=True, numéricas se imputan(mediana)+estandarizan dentro de cada fold (sin fuga).
    Embeddings del asunto con PCA por fold cuando use_emb=True."""
    num_cols = [f for f in feats if f not in CAT_COLS]
    cat_cols = [f for f in feats if f in CAT_COLS]
    Xn = df[num_cols].values.astype(float)
    Xc = pd.get_dummies(df[cat_cols].astype("category")).values.astype(float) if cat_cols else np.zeros((len(df),0))
    oof = np.zeros(len(y))
    cv = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    for tr, va in cv.split(Xn, y, groups):
        if use_emb:
            prods_tr = np.unique(ROW_PROD[tr]); k = min(N_EMB, len(prods_tr)-1)
            pca = PCA(n_components=k, random_state=42).fit(EMB_RAW_MAT[prods_tr])
            emb = pca.transform(EMB_RAW_MAT)[ROW_PROD]; etr, eva = emb[tr], emb[va]
        else:
            etr = np.zeros((len(tr),0)); eva = np.zeros((len(va),0))
        ntr, nva = Xn[tr], Xn[va]
        if scale:
            imp = SimpleImputer(strategy="median").fit(ntr)
            ss = StandardScaler().fit(imp.transform(ntr))
            ntr = ss.transform(imp.transform(ntr)); nva = ss.transform(imp.transform(nva))
            if use_emb:
                se = StandardScaler().fit(etr); etr = se.transform(etr); eva = se.transform(eva)
        Xtr = np.hstack([ntr, Xc[tr], etr]); Xva = np.hstack([nva, Xc[va], eva])
        m = make_model(); m.fit(Xtr, y[tr]); oof[va] = m.predict_proba(Xva)[:, 1]
    return oof

ALGOS = {
    "LightGBM":             (lambda: lgb.LGBMClassifier(**LGBM_PARAMS), False),
    "Red neuronal (MLP)":   (lambda: MLPClassifier(hidden_layer_sizes=(64,32), alpha=1e-3, batch_size=256,
                                max_iter=300, early_stopping=True, n_iter_no_change=10, random_state=42), True),
    "HistGradientBoosting": (lambda: HistGradientBoostingClassifier(learning_rate=0.05, max_iter=300, random_state=42), False),
    "Random Forest":        (lambda: RandomForestClassifier(n_estimators=300, min_samples_leaf=20, n_jobs=-1, random_state=42), True),
    "Regresion logistica":  (lambda: LogisticRegression(max_iter=1000), True),
}

thr = float(y.mean())
def metricas(oof):
    pred = (oof >= thr).astype(int)
    return {"PR-AUC": round(average_precision_score(y, oof), 4),
            "ROC-AUC": round(roc_auc_score(y, oof), 4),
            "Brier": round(brier_score_loss(y, oof), 4),
            "LogLoss": round(log_loss(y, oof), 4),
            "F1": round(f1_score(y, pred), 4),
            "BalAcc": round(balanced_accuracy_score(y, pred), 4),
            "MCC": round(matthews_corrcoef(y, pred), 4)}

res_plano, res_hier, deltas = [], [], []
store = {"id_event": df["id_event"].values, "target": y}
for nombre, (mk, sc) in ALGOS.items():
    p_cond = oof_generic(feats_flat, True,  mk, sc)   # PLANO (nivel producto)
    p_sec  = oof_generic(feats_sec,  False, mk, sc)   # nivel sector
    hier   = p_sec * p_cond                            # JERÁRQUICO
    store[nombre + "|plano"] = np.round(p_cond, 6)
    store[nombre + "|jerarquico"] = np.round(hier, 6)
    mp, mh = metricas(p_cond), metricas(hier)
    res_plano.append({"Algoritmo": nombre, **mp})
    res_hier.append({"Algoritmo": nombre, **mh})
    deltas.append({"Algoritmo": nombre, "PR-AUC plano": mp["PR-AUC"],
                   "PR-AUC jerarquico": mh["PR-AUC"], "Delta (jer-plano)": round(mh["PR-AUC"]-mp["PR-AUC"], 4)})

pd.DataFrame(store).to_csv(PROCESSED_PATH / "m2_algo_oof.csv", index=False)
df_plano = pd.DataFrame(res_plano).sort_values("PR-AUC", ascending=False)
df_hier  = pd.DataFrame(res_hier).sort_values("PR-AUC", ascending=False)
df_delta = pd.DataFrame(deltas).sort_values("PR-AUC plano", ascending=False)

print("########## MODELO PLANO (PR-AUC, GroupKFold por usuario; umbral F1/BalAcc/MCC = %.3f) ##########" % thr)
print(df_plano.to_string(index=False))
print("\n########## MODELO JERARQUICO (p_sector x p_producto) ##########")
print(df_hier.to_string(index=False))
print("\n########## EFECTO DE LA JERARQUIA (Delta PR-AUC = jerarquico - plano) ##########")
print(df_delta.to_string(index=False))
print("\nBrier y LogLoss: menor = mejor. El resto: mayor = mejor.")


## 18 · Ajuste de hiperparámetros por algoritmo (comparación justa) — PENDIENTE DE EJECUTAR

Para que la comparación entre algoritmos sea **justa** (y no penalice a la red neuronal por usar
configuraciones por defecto), se ajustan los hiperparámetros de cada modelo mediante una búsqueda en
rejilla acotada, sobre las **mismas particiones GroupKFold** y el mismo modelo plano. **Aviso:** es costoso
(~30–60 min: cada configuración reentrena 5 folds). Tras ejecutarlo, actualizar la tabla de comparación de
la memoria con los números tuneados y la frase "tras ajustar los hiperparámetros, el resultado se mantiene".

In [ ]:
# ============================================================
# 18 · Ajuste de hiperparametros por algoritmo (comparacion JUSTA)   [PENDIENTE DE EJECUTAR]
# Reutiliza oof_generic / metricas / feats_flat de la seccion 17. COSTOSO (~30-60 min).
# ============================================================
from itertools import product

def mejor_config(maker, grid, scale):
    """Evalua cada combinacion de la rejilla con oof_generic (PR-AUC) y devuelve la mejor."""
    best = (-1.0, None, None)
    nombres = list(grid.keys())
    for combo in product(*grid.values()):
        params = dict(zip(nombres, combo))
        oof = oof_generic(feats_flat, True, lambda: maker(params), scale)
        ap = average_precision_score(y, oof)
        if ap > best[0]:
            best = (ap, params, oof)
    return best

REJILLAS = {
    "LightGBM": (
        lambda p: lgb.LGBMClassifier(objective="binary", random_state=42, verbose=-1, n_jobs=-1,
                                     subsample=0.8, colsample_bytree=0.8, min_child_samples=20, **p),
        {"num_leaves": [15, 31, 63], "learning_rate": [0.03, 0.05, 0.1], "n_estimators": [200, 400]}, False),
    "Red neuronal (MLP)": (
        lambda p: MLPClassifier(max_iter=300, early_stopping=True, n_iter_no_change=10,
                                batch_size=256, random_state=42, **p),
        {"hidden_layer_sizes": [(64, 32), (128, 64), (64,)], "alpha": [1e-4, 1e-3, 1e-2]}, True),
    "HistGradientBoosting": (
        lambda p: HistGradientBoostingClassifier(random_state=42, **p),
        {"learning_rate": [0.03, 0.05, 0.1], "max_iter": [200, 400], "max_leaf_nodes": [15, 31]}, False),
    "Random Forest": (
        lambda p: RandomForestClassifier(n_jobs=-1, random_state=42, **p),
        {"n_estimators": [300, 600], "max_depth": [None, 10, 20], "min_samples_leaf": [5, 20]}, True),
    "Regresion logistica": (
        lambda p: LogisticRegression(max_iter=1000, **p),
        {"C": [0.1, 1.0, 10.0]}, True),
}

filas_tuned = []
for nombre, (maker, grid, sc) in REJILLAS.items():
    ap, params, oof = mejor_config(maker, grid, sc)
    filas_tuned.append({"Algoritmo": nombre, **metricas(oof), "mejores_params": params})
tabla_tuned = pd.DataFrame(filas_tuned).sort_values("PR-AUC", ascending=False)
print("=== Comparacion TUNEADA (mejor config por algoritmo, modelo plano) ===")
print(tabla_tuned.to_string(index=False))
